# Financial linking and abnormal return extraction

This notebook takes the completed textual-analysis output and the daily stock return data, then:

1. loads and cleans the three input files
2. links each earnings call to a `PERMNO` using `ticker`
3. computes daily abnormal returns as `DlyRet - vwretd`
4. extracts short-run `CAR[-1,+3]` around each earnings call
5. extracts a longer-run abnormal return window up to the next earnings call
6. merges the financial variables back into the textual-analysis table
7. exports a regression-ready dataset

**Input files expected in the same folder as this notebook**
- `textual_analysis_results.csv`
- `daily_ret.csv`
- `linking_table.csv`

In [24]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. File paths

In [25]:
from pathlib import Path

BASE = Path("../../data")

TA_RES_PATH = BASE / "processed/textual_analysis_results.csv"
RET_PATH = BASE / "raw/daily_ret.csv"
LINK_PATH = BASE / "raw/linking_table.csv"

print(TA_RES_PATH.resolve())
print(RET_PATH.resolve())
print(LINK_PATH.resolve())
print("TA exists:", TA_RES_PATH.exists())
print("RET exists:", RET_PATH.exists())
print("LINK exists:", LINK_PATH.exists())

/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/textual_analysis_results.csv
/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/raw/daily_ret.csv
/Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/raw/linking_table.csv
TA exists: True
RET exists: True
LINK exists: True


## 2. Load datasets

Note: `textual_analusis_results.csv` is semicolon-separated, and several sentiment/intensity variables use commas as decimal separators.

In [26]:
ta = pd.read_csv(TA_RES_PATH, sep=";")
ret = pd.read_csv(RET_PATH)
link = pd.read_csv(LINK_PATH)

print("ta shape  :", ta.shape)
print("ret shape :", ret.shape)
print("link shape:", link.shape)

display(ta.head())
display(ret.head())
display(link.head())

ta shape  : (1547, 37)
ret shape : (102309, 11)
link shape: (102, 7)


,company,ticker,date,file,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa
0,3M Co,MMM,2022-04-26,2022-Apr-26-MMM.N-140622159458-Transcript.txt,11615,5274,6153,0,0,0,9,9,0,134,174,111,"-0,12987012987012986","0,010285396590066716",88,79,40,"0,05389221556886228","0,008183306055646482",46,94,71,"-0,34285714285714286","0,012354271793979467","1,0","2022,0",Q1 2022,"0,0","0,0","0,0","0,7748600947051227","1,7064846416382253","0,0"
1,3M Co,MMM,2022-01-25,2022-Jan-25-MMM.N-140663487123-Transcript.txt,11937,4250,7495,0,0,0,9,4,5,194,143,93,"0,1513353115727003","0,008355795148247979",99,59,23,"0,25316455696202533","0,005873340143003065",95,83,70,"0,06741573033707865","0,009924854671770877","4,0","2021,0",Q4 2021,"0,0","0,0","0,0","0,7539582809751194","0,9411764705882353","0,66711140760507"
2,3M Co,MMM,2022-07-26,2022-Jul-26-MMM.N-140270570993-Transcript.txt,13203,5024,7976,0,0,0,8,7,1,226,171,137,"0,1385390428211587","0,01105641191187152",143,65,50,"0,375","0,010755001075500108",83,105,87,"-0,11702127659574468","0,011492734478203434","2,0","2022,0",Q2 2022,"0,0","0,0","0,0","0,6059228963114444","1,3933121019108279","0,12537612838515547"
3,3M Co,MMM,2022-10-25,2022-Oct-25-MMM.N-140586063160-Transcript.txt,10815,4276,6369,0,0,0,25,7,18,168,114,103,"0,19148936170212766","0,010216226939099385",78,53,44,"0,19083969465648856","0,011253196930946292",90,60,59,"0,2","0,009781167108753316","3,0","2022,0",Q3 2022,"0,0","0,0","0,0","2,311604253351826","1,637043966323667","2,8261893546867642"
4,3M Co,MMM,2023-04-25,2023-Apr-25-MMM.N-136949071306-Transcript.txt,10820,3806,6832,0,0,0,9,6,3,166,117,92,"0,17314487632508835","0,009064932505665584",68,55,33,"0,10569105691056911","0,009345794392523364",98,61,59,"0,23270440251572327","0,00912465202598206","1,0","2023,0",Q1 2023,"0,0","0,0","0,0","0,8317929759704251","1,576458223857068","0,43911007025761123"


,PERMNO,HdrCUSIP,Ticker,PERMCO,IssuerNm,DlyCalDt,DlyPrc,DlyRet,DlyVol,vwretd,sprtrn
0,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-03,87.90,0.007912,10653484,0.006155,0.006374
1,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-04,88.84,0.010694,11958975,-0.002351,-0.000630
2,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-05,86.46,-0.026790,11236680,-0.021879,-0.019393
3,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-06,86.34,0.002313,7918392,0.000161,-0.000964
4,10104,68389X10,ORCL,8045,ORACLE CORP,2022-01-07,87.51,0.013551,9836795,-0.004164,-0.004050


,PERMNO,HdrCUSIP,CUSIP,Ticker,PERMCO,IssuerNm,DlyCalDt
0,10104,68389X10,68389X10,ORCL,8045,ORACLE CORP,2025-12-31
1,10107,59491810,59491810,MSFT,8048,MICROSOFT CORP,2025-12-31
2,10145,43851610,43851610,HON,22168,HONEYWELL INTERNATIONAL INC,2025-12-31
3,11308,19121610,19121610,KO,20468,COCA COLA CO,2025-12-31
4,11850,30231G10,30231G10,XOM,20678,EXXON MOBIL CORP,2025-12-31


## 3. Basic cleaning

In [27]:
# Dates
ta["date"] = pd.to_datetime(ta["date"], errors="coerce")
ret["DlyCalDt"] = pd.to_datetime(ret["DlyCalDt"], errors="coerce")

# Convert comma-decimal columns in textual_analysis_results.csv to numeric
comma_decimal_cols = [
    "lm_tone_total", "lm_uncertainty_total",
    "lm_tone_pres", "lm_uncertainty_pres",
    "lm_tone_qa", "lm_uncertainty_qa",
    "core_per_1000_total", "core_per_1000_pres", "core_per_1000_qa",
    "adj_per_1000_total", "adj_per_1000_pres", "adj_per_1000_qa",
]

for col in comma_decimal_cols:
    if col in ta.columns:
        ta[col] = (
            ta[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .replace("nan", np.nan)
            .astype(float)
        )

# Ticker cleanup
ta["ticker"] = ta["ticker"].astype(str).str.upper().str.strip()
link["Ticker"] = link["Ticker"].astype(str).str.upper().str.strip()

# PERMNO / returns cleanup
ret["PERMNO"] = pd.to_numeric(ret["PERMNO"], errors="coerce").astype("Int64")
link["PERMNO"] = pd.to_numeric(link["PERMNO"], errors="coerce").astype("Int64")
ret["DlyRet"] = pd.to_numeric(ret["DlyRet"], errors="coerce")
ret["vwretd"] = pd.to_numeric(ret["vwretd"], errors="coerce")

print(ta.dtypes.head(20))

company                               object
ticker                                object
date                          datetime64[ns]
file                                  object
total_words                            int64
pres_words                             int64
qa_words                               int64
core_hits_total                        int64
core_hits_pres                         int64
core_hits_qa                           int64
adj_hits_total                         int64
adj_hits_pres                          int64
adj_hits_qa                            int64
lm_positive_total                      int64
lm_negative_total                      int64
lm_uncertainty_count_total             int64
lm_tone_total                        float64
lm_uncertainty_total                 float64
lm_positive_pres                       int64
lm_negative_pres                       int64
dtype: object


## 4. Linking Table

In [28]:
# Keep one row per ticker/PERMNO pair in link table
link_unique = (
    link[["PERMNO", "Ticker", "IssuerNm"]]
    .drop_duplicates()
    .copy()
)

display(link_unique.head())

,PERMNO,Ticker,IssuerNm
0,10104,ORCL,ORACLE CORP
1,10107,MSFT,MICROSOFT CORP
2,10145,HON,HONEYWELL INTERNATIONAL INC
3,11308,KO,COCA COLA CO
4,11850,XOM,EXXON MOBIL CORP


## 5. Link earnings calls to PERMNO

In [29]:
ta_linked = ta.merge(
    link_unique,
    left_on="ticker",
    right_on="Ticker",
    how="left"
)

print("Rows without PERMNO after ticker merge:", ta_linked["PERMNO"].isna().sum())
display(
    ta_linked[ta_linked["PERMNO"].isna()][["company", "ticker", "file"]].drop_duplicates()
)
display(ta_linked.head())

Rows without PERMNO after ticker merge: 0


,company,ticker,file


,company,ticker,date,file,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa,PERMNO,Ticker,IssuerNm
0,3M Co,MMM,2022-04-26,2022-Apr-26-MMM.N-140622159458-Transcript.txt,11615,5274,6153,0,0,0,9,9,0,134,174,111,-0.129870,0.010285,88,79,40,0.053892,0.008183,46,94,71,-0.342857,0.012354,"1,0","2022,0",Q1 2022,0.0,0.0,0.0,0.774860,1.706485,0.000000,22592,MMM,3M CO
1,3M Co,MMM,2022-01-25,2022-Jan-25-MMM.N-140663487123-Transcript.txt,11937,4250,7495,0,0,0,9,4,5,194,143,93,0.151335,0.008356,99,59,23,0.253165,0.005873,95,83,70,0.067416,0.009925,"4,0","2021,0",Q4 2021,0.0,0.0,0.0,0.753958,0.941176,0.667111,22592,MMM,3M CO
2,3M Co,MMM,2022-07-26,2022-Jul-26-MMM.N-140270570993-Transcript.txt,13203,5024,7976,0,0,0,8,7,1,226,171,137,0.138539,0.011056,143,65,50,0.375000,0.010755,83,105,87,-0.117021,0.011493,"2,0","2022,0",Q2 2022,0.0,0.0,0.0,0.605923,1.393312,0.125376,22592,MMM,3M CO
3,3M Co,MMM,2022-10-25,2022-Oct-25-MMM.N-140586063160-Transcript.txt,10815,4276,6369,0,0,0,25,7,18,168,114,103,0.191489,0.010216,78,53,44,0.190840,0.011253,90,60,59,0.200000,0.009781,"3,0","2022,0",Q3 2022,0.0,0.0,0.0,2.311604,1.637044,2.826189,22592,MMM,3M CO
4,3M Co,MMM,2023-04-25,2023-Apr-25-MMM.N-136949071306-Transcript.txt,10820,3806,6832,0,0,0,9,6,3,166,117,92,0.173145,0.009065,68,55,33,0.105691,0.009346,98,61,59,0.232704,0.009125,"1,0","2023,0",Q1 2023,0.0,0.0,0.0,0.831793,1.576458,0.439110,22592,MMM,3M CO


**Sanity check to see if each ticker really only equals 1 PERMNO**

In [30]:
print("Unique tickers in textual results:", ta["ticker"].nunique())
print("Unique tickers matched to PERMNO:", ta_linked.loc[ta_linked["PERMNO"].notna(), "ticker"].nunique())

display(
    ta_linked.groupby("ticker", dropna=False)["PERMNO"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

Unique tickers in textual results: 98
Unique tickers matched to PERMNO: 98


ticker
AAPL    1
PG      1
PEP     1
ORCL    1
NVDA    1
NOW     1
NKE     1
NFLX    1
NEE     1
MSFT    1
MS      1
MRK     1
MO      1
MMM     1
META    1
MET     1
MDT     1
MDLZ    1
MCD     1
MA      1
Name: PERMNO, dtype: int64

## 6. Compute daily abnormal return

Using your chosen market-adjusted specification:

`AR_{i,t} = DlyRet_{i,t} - vwretd_t`

In [31]:
ret = ret.copy()
ret["abret"] = ret["DlyRet"] - ret["vwretd"]

# Keep only what is needed
ret_small = (
    ret[["PERMNO", "Ticker", "IssuerNm", "DlyCalDt", "DlyRet", "vwretd", "abret"]]
    .sort_values(["PERMNO", "DlyCalDt"])
    .reset_index(drop=True)
)

display(ret_small.head())

,PERMNO,Ticker,IssuerNm,DlyCalDt,DlyRet,vwretd,abret
0,10104,ORCL,ORACLE CORP,2022-01-03,0.007912,0.006155,0.001757
1,10104,ORCL,ORACLE CORP,2022-01-04,0.010694,-0.002351,0.013045
2,10104,ORCL,ORACLE CORP,2022-01-05,-0.026790,-0.021879,-0.004911
3,10104,ORCL,ORACLE CORP,2022-01-06,0.002313,0.000161,0.002152
4,10104,ORCL,ORACLE CORP,2022-01-07,0.013551,-0.004164,0.017715


## 7. Prepare event table

Sort each firm's earnings calls so we can identify the next call date for the long-run window.

In [32]:
events = ta_linked.copy()
events = events.sort_values(["PERMNO", "date"]).reset_index(drop=True)
events["next_call_date"] = events.groupby("PERMNO")["date"].shift(-1)

display(events[["company", "ticker", "date", "PERMNO", "next_call_date"]].head(10))

,company,ticker,date,PERMNO,next_call_date
0,Oracle Corp,ORCL,2022-03-10,10104,2022-06-13
1,Oracle Corp,ORCL,2022-06-13,10104,2022-09-12
2,Oracle Corp,ORCL,2022-09-12,10104,2022-12-12
3,Oracle Corp,ORCL,2022-12-12,10104,2023-03-09
4,Oracle Corp,ORCL,2023-03-09,10104,2023-06-12
5,Oracle Corp,ORCL,2023-06-12,10104,2023-09-11
6,Oracle Corp,ORCL,2023-09-11,10104,2023-12-11
7,Oracle Corp,ORCL,2023-12-11,10104,2024-03-11
8,Oracle Corp,ORCL,2024-03-11,10104,2024-06-11
9,Oracle Corp,ORCL,2024-06-11,10104,2024-09-09


## 8. Helper functions for event-window extraction

Two important choices here:

- `CAR[-1,+3]` uses **trading days**, not calendar days.
- If the earnings call date itself is not a trading day, the notebook shifts to the **next available trading day** for that firm.

In [33]:
def get_trading_dates_for_firm(df_firm):
    return df_firm["DlyCalDt"].dropna().drop_duplicates().sort_values().tolist()

def get_event_trading_day(df_firm, event_date):
    # first trading day on or after the event date
    dates = df_firm.loc[df_firm["DlyCalDt"] >= event_date, "DlyCalDt"].drop_duplicates().sort_values()
    if len(dates) == 0:
        return pd.NaT
    return dates.iloc[0]

def compute_car_minus1_plus3(df_firm, event_date):
    event_trading_day = get_event_trading_day(df_firm, event_date)
    if pd.isna(event_trading_day):
        return pd.Series({
            "event_trading_day": pd.NaT,
            "car_m1_p3": np.nan,
            "n_days_car": 0
        })

    firm_dates = df_firm["DlyCalDt"].drop_duplicates().sort_values().reset_index(drop=True)
    try:
        idx = firm_dates[firm_dates == event_trading_day].index[0]
    except IndexError:
        return pd.Series({
            "event_trading_day": pd.NaT,
            "car_m1_p3": np.nan,
            "n_days_car": 0
        })

    start_idx = max(idx - 1, 0)
    end_idx = min(idx + 3, len(firm_dates) - 1)
    window_dates = firm_dates.iloc[start_idx:end_idx + 1]

    temp = df_firm[df_firm["DlyCalDt"].isin(window_dates)].copy()

    return pd.Series({
        "event_trading_day": event_trading_day,
        "car_m1_p3": temp["abret"].sum(skipna=True),
        "n_days_car": temp["abret"].notna().sum()
    })

def compute_long_run_abret(df_firm, event_date, next_call_date):
    event_trading_day = get_event_trading_day(df_firm, event_date)
    if pd.isna(event_trading_day):
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    # Start the day after the event trading day
    firm_dates = df_firm["DlyCalDt"].drop_duplicates().sort_values().reset_index(drop=True)
    try:
        event_idx = firm_dates[firm_dates == event_trading_day].index[0]
    except IndexError:
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    start_idx = event_idx + 1
    if start_idx >= len(firm_dates):
        return pd.Series({
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    long_run_start = firm_dates.iloc[start_idx]

    # If there is a next call, end at the trading day before that next call's event trading day
    if pd.notna(next_call_date):
        next_event_trading_day = get_event_trading_day(df_firm, next_call_date)
        if pd.notna(next_event_trading_day):
            next_idx = firm_dates[firm_dates == next_event_trading_day].index[0]
            end_idx = next_idx - 1
        else:
            end_idx = len(firm_dates) - 1
    else:
        end_idx = len(firm_dates) - 1

    if end_idx < start_idx:
        return pd.Series({
            "long_run_start": long_run_start,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })

    window_dates = firm_dates.iloc[start_idx:end_idx + 1]
    long_run_end = window_dates.iloc[-1]

    temp = df_firm[df_firm["DlyCalDt"].isin(window_dates)].copy()

    return pd.Series({
        "long_run_start": long_run_start,
        "long_run_end": long_run_end,
        "long_run_abret": temp["abret"].sum(skipna=True),
        "n_days_long_run": temp["abret"].notna().sum()
    })

## 9. Run the extraction firm by firm

In [34]:
results = []

for _, row in events.iterrows():
    permno = row["PERMNO"]
    event_date = row["date"]
    next_call_date = row["next_call_date"]

    if pd.isna(permno) or pd.isna(event_date):
        out = row.to_dict()
        out.update({
            "event_trading_day": pd.NaT,
            "car_m1_p3": np.nan,
            "n_days_car": 0,
            "long_run_start": pd.NaT,
            "long_run_end": pd.NaT,
            "long_run_abret": np.nan,
            "n_days_long_run": 0
        })
        results.append(out)
        continue

    df_firm = ret_small[ret_small["PERMNO"] == permno].sort_values("DlyCalDt").copy()

    car_info = compute_car_minus1_plus3(df_firm, event_date)
    long_info = compute_long_run_abret(df_firm, event_date, next_call_date)

    out = row.to_dict()
    out.update(car_info.to_dict())
    out.update(long_info.to_dict())
    results.append(out)

financial_event_df = pd.DataFrame(results)

display(
    financial_event_df[
        ["company", "date", "PERMNO", "event_trading_day", "car_m1_p3", "n_days_car", "long_run_start", "long_run_end", "long_run_abret", "n_days_long_run"]
    ].head(15)
)

,company,date,PERMNO,event_trading_day,car_m1_p3,n_days_car,long_run_start,long_run_end,long_run_abret,n_days_long_run
0,Oracle Corp,2022-03-10,10104,2022-03-10,0.079489,5,2022-03-11,2022-06-10,-0.034878,64
1,Oracle Corp,2022-06-13,10104,2022-06-13,0.091076,5,2022-06-14,2022-09-09,0.096944,61
2,Oracle Corp,2022-09-12,10104,2022-09-12,-0.039462,5,2022-09-13,2022-12-09,0.089863,63
3,Oracle Corp,2022-12-12,10104,2022-12-12,0.017250,5,2022-12-13,2023-03-08,0.077589,58
4,Oracle Corp,2023-03-09,10104,2023-03-09,-0.021471,5,2023-03-10,2023-06-09,0.162663,64
5,Oracle Corp,2023-06-12,10104,2023-06-12,0.139119,5,2023-06-13,2023-09-08,0.066229,61
6,Oracle Corp,2023-09-11,10104,2023-09-11,-0.097147,5,2023-09-12,2023-12-08,-0.117493,63
7,Oracle Corp,2023-12-11,10104,2023-12-11,-0.143001,5,2023-12-12,2024-03-08,-0.109181,60
8,Oracle Corp,2024-03-11,10104,2024-03-11,0.101778,5,2024-03-12,2024-06-10,0.062857,63
9,Oracle Corp,2024-06-11,10104,2024-06-11,0.089616,5,2024-06-12,2024-09-06,0.137404,60


## 10. Quick quality checks

In [35]:
print("Missing PERMNO rows:", financial_event_df["PERMNO"].isna().sum())
print("Missing CAR rows   :", financial_event_df["car_m1_p3"].isna().sum())
print("Missing long-run rows:", financial_event_df["long_run_abret"].isna().sum())

print("\nDistribution of n_days_car")
print(financial_event_df["n_days_car"].value_counts(dropna=False).sort_index())

print("\nDistribution of n_days_long_run")
print(financial_event_df["n_days_long_run"].value_counts(dropna=False).sort_index().head(20))

display(financial_event_df[financial_event_df["n_days_car"] < 4][["company", "date", "PERMNO", "event_trading_day", "n_days_car"]].head(20))

Missing PERMNO rows: 0
Missing CAR rows   : 0
Missing long-run rows: 0

Distribution of n_days_car
n_days_car
5    1547
Name: count, dtype: int64

Distribution of n_days_long_run
n_days_long_run
2      1
8      3
13     2
14     2
19     1
23     1
27     2
28     3
29     2
32     1
33     1
36     1
37     2
38     4
39     4
40     2
41     5
42    12
43    11
44     8
Name: count, dtype: int64


,company,date,PERMNO,event_trading_day,n_days_car


For `CAR[-1,+3]`, you will usually expect **5 trading days** when the firm has full return coverage for the window.  
If some rows show fewer days, inspect them manually.

## 11. Create final regression-ready dataset

In [36]:
final_cols_front = [
    "company", "ticker", "date", "file", "PERMNO", "Ticker", "IssuerNm",
    "event_trading_day", "next_call_date",
    "car_m1_p3", "n_days_car",
    "long_run_start", "long_run_end", "long_run_abret", "n_days_long_run"
]

remaining_cols = [c for c in financial_event_df.columns if c not in final_cols_front]
final_df = financial_event_df[final_cols_front + remaining_cols].copy()

display(final_df.head())
print(final_df.shape)

,company,ticker,date,file,PERMNO,Ticker,IssuerNm,event_trading_day,next_call_date,car_m1_p3,n_days_car,long_run_start,long_run_end,long_run_abret,n_days_long_run,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,adj_hits_total,adj_hits_pres,adj_hits_qa,lm_positive_total,lm_negative_total,lm_uncertainty_count_total,lm_tone_total,lm_uncertainty_total,lm_positive_pres,lm_negative_pres,lm_uncertainty_count_pres,lm_tone_pres,lm_uncertainty_pres,lm_positive_qa,lm_negative_qa,lm_uncertainty_count_qa,lm_tone_qa,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa
0,Oracle Corp,ORCL,2022-03-10,2022-Mar-10-ORCL.N-139346710200-Transcript.txt,10104,ORCL,ORACLE CORP,2022-03-10,2022-06-13,0.079489,5,2022-03-11,2022-06-10,-0.034878,64,7503,4184,3197,0,0,0,1,1,0,103,41,47,0.430556,0.006636,76,19,23,0.600000,0.005884,27,21,24,0.125000,0.007807,"3,0","2022,0",Q3 2022,0.000000,0.000000,0.000000,0.133280,0.239006,0.000000
1,Oracle Corp,ORCL,2022-06-13,2022-Jun-13-ORCL.N-140185486493-Transcript.txt,10104,ORCL,ORACLE CORP,2022-06-13,2022-09-12,0.091076,5,2022-06-14,2022-09-09,0.096944,61,6921,3452,3333,1,1,0,3,3,0,79,39,45,0.338983,0.006904,42,12,22,0.555556,0.006875,37,26,23,0.174603,0.007174,"4,0","2022,0",Q4 2022,0.144488,0.289687,0.000000,0.433463,0.869061,0.000000
2,Oracle Corp,ORCL,2022-09-12,2022-Sep-12-ORCL.N-138707918626-Transcript.txt,10104,ORCL,ORACLE CORP,2022-09-12,2022-12-12,-0.039462,5,2022-09-13,2022-12-09,0.089863,63,6634,3055,3448,7,7,0,1,0,1,59,47,52,0.113208,0.008321,23,11,23,0.352941,0.008200,36,35,29,0.014085,0.008690,"1,0","2023,0",Q1 2023,1.055170,2.291326,0.000000,0.150739,0.000000,0.290023
3,Oracle Corp,ORCL,2022-12-12,2022-Dec-12-ORCL.N-140500431181-Transcript.txt,10104,ORCL,ORACLE CORP,2022-12-12,2023-03-09,0.017250,5,2022-12-13,2023-03-08,0.077589,58,7684,4586,2974,13,8,5,8,5,3,68,42,43,0.236364,0.005929,40,9,20,0.632653,0.004662,28,32,23,-0.066667,0.008036,"2,0","2023,0",Q2 2023,1.691827,1.744440,1.681237,1.041124,1.090275,1.008742
4,Oracle Corp,ORCL,2023-03-09,2023-Mar-09-ORCL.N-138300942700-Transcript.txt,10104,ORCL,ORACLE CORP,2023-03-09,2023-06-12,-0.021471,5,2023-03-10,2023-06-09,0.162663,64,7267,3626,3515,45,8,37,1,0,1,72,39,45,0.297297,0.006555,38,12,20,0.520000,0.005981,34,26,25,0.133333,0.007316,"3,0","2023,0",Q3 2023,6.192376,2.206288,10.526316,0.137608,0.000000,0.284495


(1547, 48)


## 12. Export outputs

In [37]:
# output directory
OUTPUT_DIR = Path("../../data/processed")

OUTPUT_EVENTS = OUTPUT_DIR / "financial_event_dataset.csv"
OUTPUT_RETURNS = OUTPUT_DIR / "daily_returns_with_abret.csv"

final_df.to_csv(OUTPUT_EVENTS, index=False)
ret_small.to_csv(OUTPUT_RETURNS, index=False)

print("Saved:", OUTPUT_EVENTS.resolve())
print("Saved:", OUTPUT_RETURNS.resolve())

Saved: /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/financial_event_dataset.csv
Saved: /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/daily_returns_with_abret.csv


## 13. Optional: very small diagnostic sample

Use this to inspect one firm manually and verify the event windows.

In [23]:
sample_firm = final_df["company"].dropna().iloc[0]
sample_permno = final_df.loc[final_df["company"] == sample_firm, "PERMNO"].iloc[0]

print("Sample firm :", sample_firm)
print("Sample PERMNO:", sample_permno)

display(final_df[final_df["company"] == sample_firm][[
    "company", "date", "event_trading_day", "car_m1_p3",
    "long_run_start", "long_run_end", "long_run_abret"
]].sort_values("date"))

Sample firm : Oracle Corp
Sample PERMNO: 10104


,company,date,event_trading_day,car_m1_p3,long_run_start,long_run_end,long_run_abret
0,Oracle Corp,2022-03-10,2022-03-10,0.079489,2022-03-11,2022-06-10,-0.034878
1,Oracle Corp,2022-06-13,2022-06-13,0.091076,2022-06-14,2022-09-09,0.096944
2,Oracle Corp,2022-09-12,2022-09-12,-0.039462,2022-09-13,2022-12-09,0.089863
3,Oracle Corp,2022-12-12,2022-12-12,0.017250,2022-12-13,2023-03-08,0.077589
4,Oracle Corp,2023-03-09,2023-03-09,-0.021471,2023-03-10,2023-06-09,0.162663
5,Oracle Corp,2023-06-12,2023-06-12,0.139119,2023-06-13,2023-09-08,0.066229
6,Oracle Corp,2023-09-11,2023-09-11,-0.097147,2023-09-12,2023-12-08,-0.117493
7,Oracle Corp,2023-12-11,2023-12-11,-0.143001,2023-12-12,2024-03-08,-0.109181
8,Oracle Corp,2024-03-11,2024-03-11,0.101778,2024-03-12,2024-06-10,0.062857
9,Oracle Corp,2024-06-11,2024-06-11,0.089616,2024-06-12,2024-09-06,0.137404
